In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision.datasets import MNIST
import torchvision.transforms as transforms

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = MNIST(root = "./data", train = True, download = True, transform = transform)
testset = MNIST(root = "./data", train = False, download = True, transform = transform)

100%|██████████| 9.91M/9.91M [00:00<00:00, 12.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 339kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.18MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.09MB/s]


In [3]:
trainLoader = DataLoader(trainset, batch_size = 64, shuffle = True)
testLoader = DataLoader(testset, batch_size = 64)

In [4]:
trainset

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5,), std=(0.5,))
           )

# Build CNN

In [5]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 28, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(28, 56, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(7 * 7 * 56, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)

        return x

In [6]:
model = CNN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Training

In [10]:
epochs = 10

for epoch in range(epochs):
    model.train()
    epoch_training_loss = 0.0

    for image, label in trainLoader:
        optimizer.zero_grad()

        output = model.forward(image)
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    model.eval()
    running_validation_loss = 0.0
    best_validation_loss = float('inf')

    with torch.no_grad():
        for image, label in testLoader:
            output = model.forward(image)
            loss = criterion(output, label)
            running_validation_loss += loss.item()

    epoch_validation_loss = running_validation_loss / len(testLoader)

    if epoch_validation_loss < best_validation_loss:
        best_validation_loss = epoch_validation_loss
        torch.save(model.state_dict(), "best.pt")

    print(f"Epoch: {epoch + 1}")
    print(f"Training Loss: {epoch_training_loss / len(trainLoader)}")
    print(f"Validation Loss: {epoch_validation_loss / len(testLoader)}\n")

Epoch: 1
Training Loss: 0.027173636322801332
Validation Loss: 0.00019111411447945642

Epoch: 2
Training Loss: 0.020207947400776593
Validation Loss: 0.00019184621830910237

Epoch: 3
Training Loss: 0.01605861638064355
Validation Loss: 0.0001849428538655239

Epoch: 4
Training Loss: 0.012967739274198564
Validation Loss: 0.00017937736608804875

Epoch: 5
Training Loss: 0.010720087742983893
Validation Loss: 0.00018431253232598384

Epoch: 6
Training Loss: 0.008269027192835387
Validation Loss: 0.0002448092452530501

Epoch: 7
Training Loss: 0.007850628683558085
Validation Loss: 0.00022826923727093418

Epoch: 8
Training Loss: 0.006115654007741176
Validation Loss: 0.00020483510276106151

Epoch: 9
Training Loss: 0.004826021691166189
Validation Loss: 0.0002797767977388791

Epoch: 10
Training Loss: 0.0068380762434306136
Validation Loss: 0.00022676651041544175



# Evaluation

In [12]:
correct_labels = 0
total_labels = 0

model.load_state_dict(torch.load("best.pt"))

for image, label in testLoader:
    output = model.forward(image)
    _, predicted = torch.max(output, 1)
    correct_labels += (predicted == label).sum().item()
    total_labels += label.size(0)

print(f"Accuracy: {correct_labels / total_labels * 100}")

Accuracy: 99.16
